In [1]:
from lab1.src.constant.fetching import RESULTS
%pip install feedparser pandas fastparquet bs4 pypdf2 pdfplumber sentence-transformers faiss-cpu transformers rapidfuzz symspellpy yake networkx matplotlib scipy seaborn textblob stanza nltk


[notice] A new release of pip is available: 25.0.1 -> 25.3
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
from lab1.src.constant.arxiv import CATEGORY
%pip install arxiv

import arxiv
import pandas as pd

def fetch_arxiv_papers(category, max_results=100):
    papers = []

    client = arxiv.Client()

    search = arxiv.Search(
        query=f"cat:{category}",
        max_results=max_results,
        sort_by=arxiv.SortCriterion.SubmittedDate,
        sort_order=arxiv.SortOrder.Descending
    )

    try:
        for result in client.results(search):
            paper = {
                "id": result.entry_id,
                "title": result.title,
                "authors": [author.name for author in result.authors],
                "summary": result.summary,
                "published": result.published.strftime("%Y-%m-%d"),
                "pdf_url": result.pdf_url
            }
            papers.append(paper)
            print(f"Загружена: {result.title[:60]}...")

    except Exception as e:
        print(f"Ошибка при загрузке: {e}")

    return papers

papers = fetch_arxiv_papers(CATEGORY, max_results=RESULTS)
print(f"Успешно загружено {len(papers)} статей")


[notice] A new release of pip is available: 25.0.1 -> 25.3
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.
Загружена: Deep Learning for Restoring MPI System Matrices Using Simula...
Загружена: MICCAI STS 2024 Challenge: Semi-Supervised Instance-Level To...
Загружена: Two-Dimensional Tomographic Reconstruction From Projections ...
Загружена: TokCom-UEP: Semantic Importance-Matched Unequal Error Protec...
Загружена: A lasso-alternative to Dijkstra's algorithm for identifying ...
Загружена: Hard Spatial Gating for Precision-Driven Brain Metastasis Se...
Загружена: Content Adaptive Encoding For Interactive Game Streaming...
Загружена: ColonAdapter: Geometry Estimation Through Foundation Model A...
Загружена: GACELLE: GPU-accelerated tools for model parameter estimatio...
Загружена: AutoRec: Accelerating Loss Recovery for Live Streaming in a ...
Загружена: When Do Domain-Specific Foundation Models Justify Their Cost...
За

In [3]:
df = pd.DataFrame(papers)
print(f"Размер DataFrame: {df.shape}")

df[['title', 'authors', 'summary']].head(3)

Размер DataFrame: (250, 6)


,title,authors,summary
0,Deep Learning for Restoring MPI System Matrice...,"[Artyom Tsanda, Sarah Reiss, Marija Boberg, To...",Magnetic particle imaging reconstructs tracer ...
1,MICCAI STS 2024 Challenge: Semi-Supervised Ins...,"[Yaqi Wang, Zhi Li, Chengyu Wu, Jun Liu, Yifan...",Orthopantomogram (OPGs) and Cone-Beam Computed...
2,Two-Dimensional Tomographic Reconstruction Fro...,"[Shreyas Jayant Grampurohit, Satish Mulleti, A...","In parallel beam computed tomography (CT), an ..."


In [4]:
df.to_parquet('eess_iv_raw.parquet.gzip', compression='gzip')
print("Данные сохранены в eess_iv_raw.parquet.gzip")

Данные сохранены в eess_iv_raw.parquet.gzip


In [5]:
def split_into_chunks(text, chunk_size=500, overlap=50):
    words = text.split()
    chunks = []
    start = 0

    while start < len(words):
        end = start + chunk_size
        chunk = words[start:end]
        chunks.append(" ".join(chunk))
        start += chunk_size - overlap

    return chunks

def create_chunks_dataframe(df, chunk_size=500, overlap=50):
    chunks_data = []

    for idx, row in df.iterrows():
        article_id = idx
        title = row['title']
        authors = ", ".join(row['authors'])
        summary = row['summary']

        summary_chunks = split_into_chunks(summary, chunk_size, overlap)

        for chunk_idx, chunk_text in enumerate(summary_chunks):
            chunks_data.append({
                'article_id': article_id,
                'chunk_id': chunk_idx,
                'title': title,
                'authors': authors,
                'chunk_text': chunk_text,
                'chunk_length': len(chunk_text.split())
            })

    return pd.DataFrame(chunks_data)

df_chunks = create_chunks_dataframe(df)
print(f"Создано {len(df_chunks)} чанков из {len(df)} статей")
print(f"Средняя длина чанка: {df_chunks['chunk_length'].mean():.1f} слов")

df_chunks.head()

Создано 250 чанков из 250 статей
Средняя длина чанка: 185.8 слов


,article_id,chunk_id,title,authors,chunk_text,chunk_length
0,0,0,Deep Learning for Restoring MPI System Matrice...,"Artyom Tsanda, Sarah Reiss, Marija Boberg, Tob...",Magnetic particle imaging reconstructs tracer ...,255
1,1,0,MICCAI STS 2024 Challenge: Semi-Supervised Ins...,"Yaqi Wang, Zhi Li, Chengyu Wu, Jun Liu, Yifan ...",Orthopantomogram (OPGs) and Cone-Beam Computed...,255
2,2,0,Two-Dimensional Tomographic Reconstruction Fro...,"Shreyas Jayant Grampurohit, Satish Mulleti, Aj...","In parallel beam computed tomography (CT), an ...",162
3,3,0,TokCom-UEP: Semantic Importance-Matched Unequa...,"Kaizheng Zhang, Zuolin Jin, Zhihang Cheng, Min...","Based on the provided LaTeX code, here is the ...",183
4,4,0,A lasso-alternative to Dijkstra's algorithm fo...,"Anqi Dong, Amirhossein Taghvaei, Tryphon T. Ge...",We revisit the problem of finding the shortest...,87


In [6]:
from sentence_transformers import SentenceTransformer
import faiss

model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')

texts_to_embed = df_chunks['chunk_text'].tolist()

print(f"Векторизация {len(texts_to_embed)} чанков...")
embeddings = model.encode(texts_to_embed, show_progress_bar=True, convert_to_numpy=True)

print(f"Размерность эмбеддингов: {embeddings.shape}")

/Users/dmitryvasilkov/PycharmProjects/labs/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Векторизация 250 чанков...


Batches: 100%|██████████| 8/8 [00:00<00:00, 20.30it/s]

Размерность эмбеддингов: (250, 384)


In [7]:
embedding_dim = embeddings.shape[1]
index = faiss.IndexFlatL2(embedding_dim)
index.add(embeddings)

print(f"FAISS индекс создан. Количество векторов: {index.ntotal}")

faiss.write_index(index, "eess_iv_index.faiss")
df_chunks.to_parquet("eess_iv_chunks.parquet.gzip", compression="gzip")

print("Векторное хранилище сохранено")

FAISS индекс создан. Количество векторов: 250
Векторное хранилище сохранено


In [8]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from symspellpy import SymSpell

sym_spell = SymSpell(max_dictionary_edit_distance=2, prefix_length=7)

print("Загрузка FLAN-T5 модели...")
flan_model = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-base")
flan_tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-base")

Загрузка FLAN-T5 модели...


In [21]:
%pip install rapidfuzz symspellpy

from symspellpy import SymSpell, Verbosity
import pkg_resources

technical_terms = [
    "deep", "learning", "machine", "neural", "network", "image", "segmentation",
    "medical", "analysis", "approaches", "summarize", "recent", "advances",
    "processing", "current", "trends", "computer", "vision", "research",
    "classification", "methods", "object", "detection", "transformer",
    "convolutional", "super", "resolution", "feature", "extraction",
    "vision", "transformer", "architectures", "attention", "mechanism",
    "convolution", "pooling", "activation", "function", "optimizer",
    "backpropagation", "gradient", "descent", "loss", "function",
    "accuracy", "precision", "recall", "f1", "score", "metric"
]

def correct_spelling(query):
    sym_spell = SymSpell(max_dictionary_edit_distance=2, prefix_length=7)

    try:
        dictionary_path = pkg_resources.resource_filename(
            "symspellpy", "frequency_dictionary_en_82_765.txt"
        )
        sym_spell.load_dictionary(dictionary_path, term_index=0, count_index=1)
    except:
        print("Используем кастомный словарь...")
        for term in technical_terms:
            sym_spell.create_dictionary_entry(term, 1000)
        common_words = ["what", "are", "for", "in", "do", "how", "work", "the", "and",
                       "with", "using", "based", "from", "this", "that", "which"]
        for word in common_words:
            sym_spell.create_dictionary_entry(word, 2000)

    suggestions = sym_spell.lookup_compound(query, max_edit_distance=2)
    if suggestions:
        return suggestions[0].term
    return query

queries_with_typos = [
    "What are deeap lerning methods for imag segmantation?",
    "Compre medical imge analysis approches",
    "Summrize recent advancs in image procesing",
    "Currnt trends in computr vision reserch",
    "How do neurnal netwoks work for image clasification?"
]

for query in queries_with_typos:
    corrected = correct_spelling(query)
    print(f"Оригинал:  '{query}'")
    print(f"Исправлено: '{corrected}'")
    print(f"Изменения: {query != corrected}")
    print("-" * 50)

def search_similar_chunks(query, top_k=3):
    query_embedding = model.encode([query], convert_to_numpy=True)
    distances, indices = index.search(query_embedding, top_k)

    results = []
    for i, idx in enumerate(indices[0]):
        chunk_data = df_chunks.iloc[idx].to_dict()
        chunk_data['distance'] = distances[0][i]
        results.append(chunk_data)

    return results

def create_rag_prompt(query, context_chunks, prompt_type="qa"):
    context_text = "\n".join([
        f"Article: {chunk['title'][:60]}...\nText: {chunk['chunk_text'][:200]}..."
        for chunk in context_chunks[:2]
    ])

    prompts = {
        "qa": f"""Answer based on these articles:

{context_text}

Question: {query}

Answer:""",

        "summary": f"""Summarize main points:

{context_text}

Summary:""",

        "comparison": f"""Compare approaches:

{context_text}

Comparison:""",

        "methods": f"""What methods are used?

{context_text}

Methods:""",

        "trends": f"""What trends are shown?

{context_text}

Trends:"""
    }

    return prompts.get(prompt_type, prompts["qa"])

def ask_flan_rag(query, prompt_type="qa", top_k=3):
    corrected_query = correct_spelling(query)
    if corrected_query != query:
        print(f"Исправлен запрос: '{query}' -> '{corrected_query}'")

    context_chunks = search_similar_chunks(corrected_query, top_k)

    prompt = create_rag_prompt(corrected_query, context_chunks, prompt_type)

    inputs = flan_tokenizer(prompt, return_tensors="pt", truncation=True, max_length=400)
    outputs = flan_model.generate(
        **inputs,
        max_new_tokens=100,
        do_sample=False,
        temperature=0.1
    )

    answer = flan_tokenizer.decode(outputs[0], skip_special_tokens=True)

    return {
        'answer': answer,
        'corrected_query': corrected_query,
        'context_chunks': context_chunks,
        'prompt_type': prompt_type
    }

def ask_flan_direct(query):
    prompt = f"Q: {query}\nA:"
    inputs = flan_tokenizer(prompt, return_tensors="pt", truncation=True, max_length=200)
    outputs = flan_model.generate(**inputs, max_new_tokens=100, do_sample=False)
    return flan_tokenizer.decode(outputs[0], skip_special_tokens=True)

test_queries = [
    {"query": "What are deeap learning methods for image segmentation", "type": "methods"},
    {"query": "Compare medical image analysis approaches", "type": "comparison"},
    {"query": "Summarize recent advances in image processing", "type": "summary"},
    {"query": "What are current trends in computer vision research", "type": "trends"},
    {"query": "How do neural networks work for image classification", "type": "qa"}
]

for i, test in enumerate(test_queries, 1):
    print(f"\n{'='*50}")
    print(f"ТЕСТ {i}: {test['type']}")
    print(f"Оригинал: '{test['query']}'")

    corrected = correct_spelling(test['query'])
    if corrected != test['query']:
        print(f"Исправлено: '{corrected}'")

    result = ask_flan_rag(test['query'], test['type'])
    print(f"\nRAG Ответ: {result['answer']}")

    direct = ask_flan_direct(test['query'])
    print(f"Без RAG: {direct}")

    print(f"\nИспользовано статей: {len(result['context_chunks'])}")

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)



[notice] A new release of pip is available: 25.0.1 -> 25.3
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.
Оригинал:  'What are deeap lerning methods for imag segmantation?'
Исправлено: 'what are deep learning methods for image segmentation'
Изменения: True
--------------------------------------------------
Оригинал:  'Compre medical imge analysis approches'
Исправлено: 'compare medical image analysis approaches'
Изменения: True
--------------------------------------------------
Оригинал:  'Summrize recent advancs in image procesing'
Исправлено: 'summarize recent advance in image processing'
Изменения: True
--------------------------------------------------
Оригинал:  'Currnt trends in computr vision reserch'
Исправлено: 'current trends in computer vision research'
Изменения: True
--------------------------------------------------
Оригинал:  'How do neurnal netwoks work for image clasification?'
Исправлено: 'how do n

In [13]:
import pickle

system_state = {
    'model': model,
    'index': index,
    'df_chunks': df_chunks,
    'flan_model': flan_model,
    'flan_tokenizer': flan_tokenizer,
    'category': CATEGORY,
    'num_papers': len(df),
    'num_chunks': len(df_chunks)
}

with open('eess_iv_rag_system.pkl', 'wb') as f:
    pickle.dump(system_state, f)

print(f"   - Категория: {CATEGORY}")
print(f"   - Количество статей: {len(df)}")
print(f"   - Количество чанков: {len(df_chunks)}")
print(f"   - Размер векторного индекса: {index.ntotal}")

   - Категория: eess.IV
   - Количество статей: 250
   - Количество чанков: 250
   - Размер векторного индекса: 250
